# 2.2 — Detecting & Handling Outliers

An outlier is a value that is far away from the rest of the data. It could be a genuine rare event or a data entry error.

> **Why it matters:** Outliers can ruin linear regression completely — one extreme value pulls the line toward itself. You must detect and handle them before training.

### Two detection methods:
- **IQR** — works on any data, especially skewed (salary, prices)
- **Z-score** — works when data is normally distributed (bell-shaped)

### Three ways to handle them:
- **Remove** the row
- **Cap** the value to the boundary
- **Log transform** the column

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = {
    'name':   ['Arwindh','Priya','Ravi','Meena','Karan','Divya','Suresh'],
    'age':    [22, 25, 23, 24, 26, 21, 23],
    'salary': [42000, 38000, 45000, 35000, 950000, 40000, 37000]
}

df = pd.DataFrame(data)
df

## 1. Visualise First

Always plot before writing detection code — a boxplot shows outliers instantly.

In [ ]:
df['salary'].plot(kind='box', title='Salary distribution')
plt.show()

df['salary'].plot(kind='hist', bins=10, title='Salary histogram')
plt.show()

## 2. IQR Method (use for skewed data)

IQR = Q3 - Q1 (the middle 50% of data)

- Lower bound = Q1 - 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR
- Anything outside = outlier

IQR ignores extreme values when calculating bounds — that's why it works on skewed data.

In [ ]:
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR}")
print(f"Lower bound: {lower:.0f}")
print(f"Upper bound: {upper:.0f}")

In [ ]:
# Find the outlier rows
outliers_iqr = df[(df['salary'] < lower) | (df['salary'] > upper)]
print("Outliers detected by IQR:")
outliers_iqr

### 1. The Math Behind the Z-Score
To calculate a Z-score manually, you use this formula:

**z = (x - μ) / σ**

* **x**: The specific data point you are checking (e.g., one person's salary).
* **μ**: The mean (average) of the entire dataset.
* **σ**: The standard deviation (how spread out the entire dataset is).

**What the result tells you:**
* A Z-score of **0** means the value is exactly the average.
* A Z-score of **1.5** means the value is **1.5** standard deviations *above* the average.
* A Z-score of **-2** means the value is **2** standard deviations *below* the average.

---

### 2. The Rule of 3 (Detecting the Outlier)
We flag an outlier if `|z| > 3` (the absolute value of z is greater than 3).

This is based on the **Empirical Rule** in statistics, which states that in a true normal distribution:
* **~68%** of data falls within **1** standard deviation (z between -1 and 1).
* **~95%** of data falls within **2** standard deviations (z between -2 and 2).
* **~99.7%** of data falls within **3** standard deviations (z between -3 and 3).

Because **99.7%** of all normal data should live between a Z-score of -3 and 3, anything that lands outside of that range (like a Z-score of 3.5 or -4) is extremely rare—so we mathematically flag it as an anomaly or outlier.

## 3. Z-Score Method (use for normally distributed data)

Z-score = how many standard deviations a value is from the mean.

- If |z| > 3 → outlier

Z-score uses the mean — so it breaks on skewed data because the mean gets pulled toward the tail.

In [ ]:
from scipy import stats

z_scores = stats.zscore(df['salary'])
print("Z-scores:", z_scores.round(2))

outliers_z = df[abs(z_scores) > 3]
print("\nOutliers detected by Z-score:")
print(outliers_z)

In [ ]:
# Without scipy — manual z-score
mean = df['salary'].mean()
std  = df['salary'].std()
df['z_score'] = (df['salary'] - mean) / std
df

## 4. Handling Outliers

### Option 1 — Remove the row

In [ ]:
df_removed = df[(df['salary'] >= lower) & (df['salary'] <= upper)]
print("After removing outliers:")
print(df_removed[['name','salary']])

### Option 2 — Cap (clip) the value
Instead of removing the row, replace the outlier with the boundary value.

In [ ]:
df_capped = df.copy()
df_capped['salary'] = df_capped['salary'].clip(lower=lower, upper=upper)
print("After capping:")
print(df_capped[['name','salary']])

### Option 3 — Log transform
Compresses large values so extreme outliers have less impact. Common for salary, price, population.

In [ ]:
df['salary_log'] = np.log1p(df['salary'])  # log1p = log(1+x), safe for zeros
print("Original vs log-transformed:")
print(df[['name','salary','salary_log']])

## 5. Decision Rules

| Situation | Strategy |
|-----------|----------|
| Data is skewed (salary, prices) | Use IQR |
| Data is normally distributed | Use Z-score |
| Outlier is clearly a data error | Remove the row |
| Outlier might be real (CEO salary) | Cap the value |
| Column has heavy right skew | Log transform |
| Using Random Forest or tree models | Can leave outliers — trees are robust |

> **Key insight:** Always visualise first. A boxplot shows you exactly where outliers are before writing a single line of detection code.

## Practice Task

In [ ]:
practice_data = {
    'name':   ['A','B','C','D','E','F','G','H'],
    'age':    [22, 25, 23, 24, 26, 21, 150, 23],
    'salary': [40000, 42000, 39000, 250000, 41000, 38000, 43000, 40500],
    'score':  [85, 78, 90, 88, -99, 76, 84, 91]
}
practice_df = pd.DataFrame(practice_data)

# Q1 — Use IQR to find outliers in salary
# YOUR CODE HERE

# Q2 — Use Z-score to find outliers in age
# YOUR CODE HERE

# Q3 — score has a value of -99 (impossible). Remove that row.
# YOUR CODE HERE

# Q4 — Cap the salary outlier instead of removing it
# YOUR CODE HERE